In [0]:
# Run Part 1 (instructor setup) first. Then run these cells in order.

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
  session_id INT,
  zone_id STRING,
  vehicle_type STRING,
  duration_hours INT,
  parking_fee INT,
  status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),
(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
  zone_id STRING,
  zone_name STRING,
  city STRING,
  capacity INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT * FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(session_id, zone_id, vehicle_type, duration_hours, parking_fee, status);

In [0]:
%sql
SELECT * FROM parking_sessions_source;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,240,Active
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
107,Z03,SUV,2,120,Cancelled
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active


In [0]:
%sql
SELECT session_id, vehicle_type, parking_fee FROM parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
SELECT * FROM parking_sessions_source
WHERE status = 'Completed' AND parking_fee > 100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
SELECT vehicle_type,
       SUM(parking_fee) AS total_fees,
       COUNT(*) AS completed_sessions
FROM parking_sessions_source
WHERE status = 'Completed'
GROUP BY vehicle_type;

vehicle_type,total_fees,completed_sessions
Car,1750,8
Bike,160,4
SUV,540,2


In [0]:
# 4
from pyspark.sql.functions import col, when

df = spark.read.table("parking_sessions_source")
df = df.withColumn("fee_category", when(col("parking_fee") >= 200, "High").otherwise("Low"))
display(df)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


## Section B

In [0]:
%sql
CREATE OR REPLACE TABLE parking_summary
USING DELTA
AS SELECT * FROM parking_sessions_source WHERE status = 'Completed';

SELECT * FROM parking_summary;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed


In [0]:
%sql
INSERT INTO parking_sessions_work VALUES (123, 'Z01', 'Car', 3, 150, 'Active');

SELECT * FROM parking_sessions_work WHERE session_id = 123;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
123,Z01,Car,3,150,Active


In [0]:
%sql
UPDATE parking_sessions_work
SET status = 'Completed', parking_fee = 280
WHERE session_id = 104;

SELECT * FROM parking_sessions_work WHERE session_id = 104;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
104,Z01,SUV,4,280,Completed


In [0]:
%sql
DELETE FROM parking_sessions_work WHERE status = 'Cancelled';

SELECT * FROM parking_sessions_work ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,280,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


## Section C

In [0]:
%sql
SELECT * FROM incoming_parking_sessions;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed
121,Z03,Car,4,200,Completed
122,Z05,Bike,2,40,Active


In [0]:
%sql
SELECT i.*
FROM incoming_parking_sessions i
INNER JOIN parking_sessions_work w ON i.session_id = w.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed


In [0]:
%sql
MERGE INTO parking_sessions_work AS t
USING incoming_parking_sessions AS s
ON t.session_id = s.session_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT * FROM parking_sessions_work ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,3,60,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,5,300,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


In [0]:
%sql
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-06T04:32:08.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(105925193984372),6c916a59-7dd2-4cea-81aa-968bead20e26,1106-041906-napwor9s-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9827, p25FileSize -> 2296, numDeletionVectorsRemoved -> 1, minFileSize -> 2296, numAddedFiles -> 1, maxFileSize -> 2296, p75FileSize -> 2296, p50FileSize -> 2296, numAddedBytes -> 2296)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T04:32:06.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(session_id#13510 = session_id#13492)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(105925193984372),6c916a59-7dd2-4cea-81aa-968bead20e26,1106-041906-napwor9s-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7534, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3593, materializeSourceTimeMs -> 340, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1211, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1966)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T04:30:50.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(105925193984372),ddc40b3b-3cb6-40ac-a46d-f86080da343d,1106-041906-napwor9s-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6105, p25FileSize -> 2293, numDeletionVectorsRemoved -> 1, minFileSize -> 2293, numAddedFiles -> 1, maxFileSize -> 2293, p75FileSize -> 2293, p50FileSize -> 2293, numAddedBytes -> 2293)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T04:30:47.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(status#12876 = Cancelled)""])",null,List(105925193984372),ddc40b3b-3cb6-40ac-a46d-f86080da343d,1106-041906-napwor9s-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1187, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 687, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 498)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T04:30:29.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(session_id#12308 = 104)""])",null,List(105925193984372),578e3c98-0fd2-4976-a002-b193d9112ed4,1106-041906-napwor9s-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4010, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1970, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1922, rewriteTimeMs -> 2009)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T04:30:04.000Z,141715606078810,azuser7411_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append,

In [0]:
# 12 - find the version just before the MERGE, then compare session 102
hist = spark.sql("DESCRIBE HISTORY parking_sessions_work")
merge_version = hist.filter("operation = 'MERGE'").agg({"version": "max"}).collect()[0][0]
before_version = merge_version - 1
print(f"MERGE version = {merge_version}, version before MERGE = {before_version}")

print("Session 102 BEFORE merge:")
display(spark.sql(f"SELECT * FROM parking_sessions_work VERSION AS OF {before_version} WHERE session_id = 102"))

print("Session 102 AFTER merge:")
display(spark.sql(f"SELECT * FROM parking_sessions_work VERSION AS OF {merge_version} WHERE session_id = 102"))

MERGE version = 5, version before MERGE = 4
Session 102 BEFORE merge:


session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,2,40,Completed


Session 102 AFTER merge:


session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed


## Section D

In [0]:
%sql
CREATE OR REPLACE VIEW completed_parking_view AS
SELECT * FROM parking_sessions_work WHERE status = 'Completed';

SELECT * FROM completed_parking_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW z01_sessions_tmp AS
SELECT * FROM parking_sessions_work WHERE zone_id = 'Z01';

SELECT * FROM z01_sessions_tmp;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
114,Z01,SUV,3,180,Completed
123,Z01,Car,3,150,Active
104,Z01,SUV,5,300,Completed


In [0]:
%sql 
CREATE OR REPLACE TEMP VIEW high_fee_sessions_gtmp AS 
SELECT * FROM parking_sessions_work 
WHERE parking_fee > 200; 

SELECT * FROM high_fee_sessions_gtmp;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
110,Z04,SUV,5,300,Active
113,Z03,Car,7,350,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed
104,Z01,SUV,5,300,Completed


In [0]:
%sql
SELECT z.zone_name, z.city, w.vehicle_type, w.parking_fee
FROM parking_sessions_work w
JOIN parking_zones z ON w.zone_id = z.zone_id
ORDER BY z.zone_name;

zone_name,city,vehicle_type,parking_fee
Airport Parking,Delhi,Car,250
Airport Parking,Delhi,Bike,40
Airport Parking,Delhi,SUV,300
Airport Parking,Delhi,Bike,20
Central Plaza,Hyderabad,Bike,80
Central Plaza,Hyderabad,SUV,180
Central Plaza,Hyderabad,Car,150
Central Plaza,Hyderabad,Car,150
Central Plaza,Hyderabad,SUV,300
City Mall,Mumbai,Car,350


In [0]:
# 16 - file count BEFORE optimize
before = spark.sql("DESCRIBE DETAIL parking_sessions_work").select("numFiles", "sizeInBytes")
print("BEFORE OPTIMIZE")
display(before)

BEFORE OPTIMIZE


numFiles,sizeInBytes
1,2296


In [0]:
%sql
OPTIMIZE parking_sessions_work;

path,metrics
abfss://unity-catalog-storage@dbstoragezi3amqsua7fge.dfs.core.windows.net/7405610695820632/__unitystorage/catalogs/c47ff2ca-0659-4a6a-9cb1-459998e4b0dd/tables/05eb370a-8e4a-4402-9dcc-1ddf6e8578a5,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793940023303, 1793940024101, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null, 0)"


In [0]:
# 16 - file count AFTER optimize
after = spark.sql("DESCRIBE DETAIL parking_sessions_work").select("numFiles", "sizeInBytes")
print("AFTER OPTIMIZE")
display(after)

# Explanation:
# OPTIMIZE compacts many small files into fewer larger ones (target ~1 GB).
# This table has only ~20 rows, so the data is tiny. It may already sit in very few files,
# or the few files it has are already far below the target size and there is nothing
# worth compacting. Hence numFiles shows little or no change. (After the DML in
# Sections B/C there may be several small files, so a drop is possible, but the
# benefit only becomes significant at larger scale.)

AFTER OPTIMIZE


numFiles,sizeInBytes
1,2296


## Section E

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS databricks_rr1.default.checkpoints;

In [0]:
# 17
rate_df = (spark.readStream
           .format("rate")
           .option("rowsPerSecond", 5)
           .load())

display(rate_df, checkpointLocation="/Volumes/databricks_rr1/default/checkpoints/q17")

Checkpointing to /Volumes/databricks_rr1/default/checkpoints/q17


In [0]:
# Q18
from pyspark.sql.functions import col

amount_df = (rate_df
             .withColumn("amount", col("value") * 10)
             .filter(col("amount") > 300))

display(amount_df, checkpointLocation="/Volumes/databricks_rr1/default/checkpoints/q18")

Checkpointing to /Volumes/databricks_rr1/default/checkpoints/q18
